In [1]:
# -*- coding: utf-8 -*-
"""WMDP Layer-by-Layer Answer Emergence Analysis"""

import torch as t
import einops
from transformer_lens import HookedTransformer, ActivationCache
from transformer_lens.utils import get_act_name
import circuitsvis as cv
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt


In [2]:
MODEL_NAME = "Qwen/Qwen3-4B"
DEVICE = 'cuda' if t.cuda.is_available() else 'cpu'

In [3]:

# Load model (using the same attention-only model from exercises, or you can use GPT2-small)
# For WMDP, we'll use a model with good reasoning capabilities
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE, dtype=t.bfloat16)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen3-4B into HookedTransformer


In [4]:


# Example WMDP-style question (multi-choice with A, B, C, D options)
# Replace this with actual WMDP questions
wmdp_question = """What is the primary mechanism by which transformer models perform in-context learning?

A) Induction heads that attend to repeated patterns
B) Positional encodings that preserve sequence order
C) Residual connections that preserve gradient flow
D) Layer normalization that stabilizes training"""

# Tokenize the input
tokens = model.to_tokens(wmdp_question)
str_tokens = model.to_str_tokens(tokens[0])

# Answer tokens (A, B, C, D with their token IDs)
# Note: GPT-2 tokenizes single letters with spaces
answer_tokens = {
    'A': model.to_tokens(" A", prepend_bos=False)[0, 0].item(),
    'B': model.to_tokens(" B", prepend_bos=False)[0, 0].item(),
    'C': model.to_tokens(" C", prepend_bos=False)[0, 0].item(),
    'D': model.to_tokens(" D", prepend_bos=False)[0, 0].item(),
}

print(f"Answer token IDs: {answer_tokens}")

def get_logits_by_layer(model, tokens):
    """Get logits at each layer by running with caching"""
    # Run with cache to get all intermediate activations
    logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)
    
    n_layers = model.cfg.n_layers
    
    # Get logits from each layer's residual stream before unembedding
    # We need to get the residual stream at each layer and apply W_U
    layer_logits = []
    
    for layer in range(n_layers):
        # Get residual stream after layer (or before unembed)
        # resid_post includes contributions up to this layer
        resid = cache["resid_post", layer]  # shape: [seq_len, d_model]
        
        # Apply unembedding to get logits
        logits_layer = resid @ model.W_U  # shape: [seq_len, d_vocab]
        layer_logits.append(logits_layer)
    
    # Also get initial logits from just embedding (layer -1)
    resid_embed = cache["embed"]
    logits_embed = resid_embed @ model.W_U
    layer_logits.insert(0, logits_embed)
    
    return layer_logits, cache

def get_answer_logits_over_layers(model, tokens, answer_tokens):
    """Extract logits for answer tokens at each layer"""
    layer_logits, _ = get_logits_by_layer(model, tokens)
    
    n_layers = len(layer_logits)  # includes embedding layer
    n_answers = len(answer_tokens)
    
    # We want the logits at the last token position (where the answer would be)
    last_token_idx = tokens.shape[1] - 1
    
    answer_logits = np.zeros((n_layers, n_answers))
    answer_names = list(answer_tokens.keys())
    
    for layer_idx, logits in enumerate(layer_logits):
        last_token_logits = logits[last_token_idx]  # shape: [d_vocab]
        for ans_idx, (ans_name, ans_token) in enumerate(answer_tokens.items()):
            answer_logits[layer_idx, ans_idx] = last_token_logits[ans_token].item()
    
    return answer_logits, answer_names


Answer token IDs: {'A': 362, 'B': 425, 'C': 356, 'D': 422}


Running model on WMDP question...


IndexError: list assignment index out of range

In [10]:
# -*- coding: utf-8 -*-
"""WMDP Layer-by-Layer Answer Emergence Analysis - Proper Unembedding at Each Layer"""

import torch as t
import einops
from transformer_lens import HookedTransformer, ActivationCache
from transformer_lens.utils import get_act_name
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import circuitsvis as cv

device = t.device("cuda" if t.cuda.is_available() else "cpu")


# Example WMDP-style question
wmdp_question = """What is the primary mechanism by which transformer models perform in-context learning?

A) Induction heads that attend to repeated patterns
B) Positional encodings that preserve sequence order
C) Residual connections that preserve gradient flow
D) Layer normalization that stabilizes training"""

# Answer tokens (A, B, C, D)
answer_tokens = {
    'A': model.to_tokens(" A", prepend_bos=False)[0, 0].item(),
    'B': model.to_tokens(" B", prepend_bos=False)[0, 0].item(),
    'C': model.to_tokens(" C", prepend_bos=False)[0, 0].item(),
    'D': model.to_tokens(" D", prepend_bos=False)[0, 0].item(),
}

print(f"Answer token IDs: {answer_tokens}")

def get_layer_unembeddings(model, tokens):
    """
    Get the unembedded logits for each layer by applying W_U to each layer's residual stream.
    Returns: list of tensors, each shape [seq_len, d_vocab]
    """
    # Run with cache to get all activations
    logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)
    
    n_layers = model.cfg.n_layers
    seq_len = tokens.shape[1]
    
    # Store unembeddings for each layer
    layer_unembeddings = []
    
    # For each layer, get the residual stream and unembed it
    for layer in range(n_layers):
        # Get residual stream AFTER this layer's attention and MLP
        # This is the representation that gets passed to the next layer
        resid_post = cache["resid_post", layer]  # shape: [seq_len, d_model]
        
        # Apply unembedding matrix W_U to get logits
        unembedded = resid_post @ model.W_U  # shape: [seq_len, d_vocab]
        layer_unembeddings.append(unembedded)
    
    # Also get unembedding from just the embedding (layer -1)
    embed = cache["embed"]  # shape: [seq_len, d_model]
    embed_unembedded = embed @ model.W_U
    layer_unembeddings.insert(0, embed_unembedded)
    
    return layer_unembeddings, cache

def extract_answer_logits(layer_unembeddings, tokens, answer_tokens):
    """
    Extract logits for answer tokens at the last position for each layer.
    """
    last_token_idx = tokens.shape[1] - 1
    n_layers = len(layer_unembeddings)
    n_answers = len(answer_tokens)
    
    answer_logits = np.zeros((n_layers, n_answers))
    answer_names = list(answer_tokens.keys())
    
    for layer_idx, unembedded in enumerate(layer_unembeddings):
        last_token_logits = unembedded[last_token_idx]  # shape: [d_vocab]
        for ans_idx, (ans_name, ans_token) in enumerate(answer_tokens.items()):
            answer_logits[layer_idx, ans_idx] = last_token_logits[ans_token].item()
    
    return answer_logits, answer_names

# Run analysis
tokens = model.to_tokens(wmdp_question)
print(f"Input shape: {tokens.shape}")
print(f"Sequence length: {tokens.shape[1]} tokens")

print("\nRunning model and extracting layer unembeddings...")
layer_unembeddings, cache = get_layer_unembeddings(model, tokens)
answer_logits, answer_names = extract_answer_logits(layer_unembeddings, tokens, answer_tokens)

# Get final prediction
final_logits = layer_unembeddings[-1][-1]  # Last layer, last token
predicted_token = final_logits.argmax().item()
predicted_str = model.to_string(predicted_token)

print(f"\nModel's prediction: '{predicted_str}'")
print(f"Answer logits at final layer:")
for name, logit in zip(answer_names, answer_logits[-1]):
    print(f"  {name}: {logit:.3f}")

# Plot 1: Raw logits through layers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Subplot 1: Raw logits
for i, name in enumerate(answer_names):
    axes[0, 0].plot(range(-1, len(answer_logits)-1), answer_logits[:, i], marker='o', label=f"Token {name}")
axes[0, 0].set_xlabel("Layer")
axes[0, 0].set_ylabel("Logit Value")
axes[0, 0].set_title("Answer Token Logits Through Layers (Raw)")
axes[0, 0].set_xticks(range(-1, len(answer_logits)-1))
axes[0, 0].set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(answer_logits)-1)])
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Subplot 2: Softmax probabilities
answer_probs = np.exp(answer_logits) / np.exp(answer_logits).sum(axis=1, keepdims=True)
for i, name in enumerate(answer_names):
    axes[0, 1].plot(range(-1, len(answer_probs)-1), answer_probs[:, i], marker='o', label=f"Token {name}")
axes[0, 1].set_xlabel("Layer")
axes[0, 1].set_ylabel("Probability")
axes[0, 1].set_title("Answer Token Probabilities Through Layers")
axes[0, 1].set_xticks(range(-1, len(answer_probs)-1))
axes[0, 1].set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(answer_probs)-1)])
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Subplot 3: Logit difference (predicted vs others)
predicted_idx = np.argmax(answer_logits[-1])
logit_diff_pred_vs_next = answer_logits[:, predicted_idx] - np.delete(answer_logits, predicted_idx, axis=1).max(axis=1)
axes[1, 0].plot(range(-1, len(logit_diff_pred_vs_next)-1), logit_diff_pred_vs_next, marker='s', color='green', linewidth=2)
axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[1, 0].set_xlabel("Layer")
axes[1, 0].set_ylabel("Logit Difference")
axes[1, 0].set_title(f"Logit({answer_names[predicted_idx]}) - Next Best")
axes[1, 0].set_xticks(range(-1, len(logit_diff_pred_vs_next)-1))
axes[1, 0].set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(logit_diff_pred_vs_next)-1)])
axes[1, 0].grid(True, alpha=0.3)

# Subplot 4: Logit difference (predicted vs mean)
logit_diff_pred_vs_mean = answer_logits[:, predicted_idx] - np.mean(answer_logits, axis=1)
axes[1, 1].plot(range(-1, len(logit_diff_pred_vs_mean)-1), logit_diff_pred_vs_mean, marker='s', color='purple', linewidth=2)
axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[1, 1].set_xlabel("Layer")
axes[1, 1].set_ylabel("Logit Difference")
axes[1, 1].set_title(f"Logit({answer_names[predicted_idx]}) - Mean of All")
axes[1, 1].set_xticks(range(-1, len(logit_diff_pred_vs_mean)-1))
axes[1, 1].set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(logit_diff_pred_vs_mean)-1)])
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot 2: Heatmap
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(answer_logits.T, cmap='RdBu_r', aspect='auto')
ax.set_xticks(range(len(answer_logits)))
ax.set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(answer_logits)-1)])
ax.set_yticks(range(len(answer_names)))
ax.set_yticklabels(answer_names)
ax.set_xlabel("Layer")
ax.set_ylabel("Answer Token")
ax.set_title("Answer Logit Heatmap Through Layers")
plt.colorbar(im, ax=ax, label="Logit Value")
plt.show()

# Advanced: Layer contribution analysis using unembeddings
def compute_layer_contributions(layer_unembeddings):
    """
    Compute how much each layer contributes to the final logits.
    Since unembeddings give us logits at each layer, we can see the incremental change.
    """
    n_layers = len(layer_unembeddings)
    contributions = []
    
    # First layer (embedding) - this is the baseline
    contributions.append(layer_unembeddings[0])
    
    # Subsequent layers - difference from previous
    for i in range(1, n_layers):
        contribution = layer_unembeddings[i] - layer_unembeddings[i-1]
        contributions.append(contribution)
    
    return contributions

contributions = compute_layer_contributions(layer_unembeddings)

# Extract answer token contributions
answer_contributions = np.zeros((len(contributions), len(answer_names)))
last_token_idx = tokens.shape[1] - 1

for layer_idx, contrib in enumerate(contributions):
    last_token_contrib = contrib[last_token_idx]  # shape: [d_vocab]
    for ans_idx, ans_token in enumerate(answer_tokens.values()):
        answer_contributions[layer_idx, ans_idx] = last_token_contrib[ans_token].item()

# Plot contributions
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(answer_contributions))
width = 0.2

for i, name in enumerate(answer_names):
    ax.bar([xi + i*width for xi in x], answer_contributions[:, i], width, label=f"Token {name}")

ax.set_xlabel("Layer")
ax.set_ylabel("Contribution to Logit")
ax.set_title("Layer Contributions to Answer Token Logits (from Unembedding)")
ax.set_xticks([xi + width*1.5 for xi in x])
ax.set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(answer_contributions)-1)])
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Cumulative contributions
cumulative_contributions = np.cumsum(answer_contributions, axis=0)
cumulative_probs = np.exp(cumulative_contributions) / np.exp(cumulative_contributions).sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 6))
for i, name in enumerate(answer_names):
    ax.plot(range(len(cumulative_probs)), cumulative_probs[:, i], marker='o', linewidth=2, label=f"Token {name}")

ax.set_xlabel("Layer")
ax.set_ylabel("Cumulative Probability")
ax.set_title("Cumulative Answer Probability Emergence (from Unembedding Contributions)")
ax.set_xticks(range(len(cumulative_probs)))
ax.set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(cumulative_probs)-1)])
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Find emergence layer
def find_emergence_layer(cumulative_probs, threshold=0.5):
    """Find first layer where an answer exceeds threshold probability"""
    for layer in range(len(cumulative_probs)):
        max_prob = cumulative_probs[layer].max()
        if max_prob > threshold:
            return layer, np.argmax(cumulative_probs[layer]), max_prob
    return None, None, None

emergence_layer, answer_idx, prob = find_emergence_layer(cumulative_probs)
if emergence_layer is not None:
    print(f"\nAnswer '{answer_names[answer_idx]}' emerges at layer {emergence_layer} (probability {prob:.3f})")

# Visualize attention at the emergence layer
if emergence_layer is not None and emergence_layer > 0:
    actual_layer = emergence_layer - 1  # Convert from cumulative index to actual layer
    if actual_layer < model.cfg.n_layers:
        attention_pattern = cache["pattern", actual_layer]
        str_tokens = model.to_str_tokens(tokens)
        
        print(f"\nAttention patterns for layer {actual_layer} (where answer emerges):")
        display(cv.attention.attention_patterns(
            tokens=str_tokens,
            attention=attention_pattern,
            attention_head_names=[f"L{actual_layer}H{i}" for i in range(model.cfg.n_heads)]
        ))

# Function to analyze multiple questions
def analyze_multiple_questions(model, questions, answer_tokens):
    """Analyze multiple WMDP questions and aggregate results"""
    results = []
    
    for q_idx, question in enumerate(tqdm(questions, desc="Analyzing questions")):
        tokens = model.to_tokens(question)
        layer_unembeddings, cache = get_layer_unembeddings(model, tokens)
        answer_logits, answer_names = extract_answer_logits(layer_unembeddings, tokens, answer_tokens)
        
        # Get probabilities
        answer_probs = np.exp(answer_logits) / np.exp(answer_logits).sum(axis=1, keepdims=True)
        
        # Find where prediction stabilizes
        final_pred = np.argmax(answer_logits[-1])
        stabilization_layer = None
        for layer in range(len(answer_probs)):
            if np.argmax(answer_probs[layer]) == final_pred:
                stabilization_layer = layer
                break
        
        results.append({
            'question_idx': q_idx,
            'final_prediction': final_pred,
            'final_prob': answer_probs[-1, final_pred],
            'stabilization_layer': stabilization_layer,
            'logit_trajectory': answer_logits,
            'prob_trajectory': answer_probs
        })
    
    return results, answer_names

# Example: Create a few variations of the question to test
questions = [
    wmdp_question,
    "The main mechanism for in-context learning in transformers is:\nA) Induction heads\nB) Positional encoding\nC) Residual connections\nD) Layer normalization",
    "What enables few-shot learning in transformer models?\nA) Attention heads that detect repeated patterns\nB) The softmax function\nC) The feedforward networks\nD) The positional embeddings"
]

# Uncomment to run multi-question analysis
# results, answer_names = analyze_multiple_questions(model, questions, answer_tokens)
# 
# # Aggregate results
# avg_stabilization = np.mean([r['stabilization_layer'] for r in results if r['stabilization_layer'] is not None])
# print(f"\nAverage answer stabilization layer: {avg_stabilization:.1f}")
# 
# # Plot distribution of stabilization layers
# stabilization_layers = [r['stabilization_layer'] for r in results if r['stabilization_layer'] is not None]
# plt.figure(figsize=(10, 5))
# plt.hist(stabilization_layers, bins=range(0, model.cfg.n_layers+2), alpha=0.7, edgecolor='black')
# plt.xlabel("Layer where answer stabilizes")
# plt.ylabel("Frequency")
# plt.title("Distribution of Answer Stabilization Layers Across Questions")
# plt.show()

print("\n=== Key Findings ===")
print(f"Final answer: {answer_names[predicted_idx]}")
print(f"Final logit: {answer_logits[-1, predicted_idx]:.3f}")
print(f"Final probability: {answer_probs[-1, predicted_idx]:.3f}")
print(f"Margin over next best: {logit_diff_pred_vs_next[-1]:.3f}")
print(f"Answer emerges at layer: {emergence_layer if emergence_layer is not None else 'Never'}")

Answer token IDs: {'A': 362, 'B': 425, 'C': 356, 'D': 422}
Input shape: torch.Size([1, 54])
Sequence length: 54 tokens

Running model and extracting layer unembeddings...


IndexError: list assignment index out of range

In [12]:
# -*- coding: utf-8 -*-
"""WMDP Layer-by-Layer Answer Emergence Analysis - Simplified Working Version"""

import torch as t
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
from tqdm import tqdm
import circuitsvis as cv

device = t.device("cuda" if t.cuda.is_available() else "cpu")



# Example WMDP-style question
wmdp_question = """What is the primary mechanism by which transformer models perform in-context learning?

A) Induction heads that attend to repeated patterns
B) Positional encodings that preserve sequence order
C) Residual connections that preserve gradient flow
D) Layer normalization that stabilizes training"""

# Answer tokens (A, B, C, D) - note the space is important
answer_tokens = {
    'A': model.to_tokens(" A", prepend_bos=False)[0, 0].item(),
    'B': model.to_tokens(" B", prepend_bos=False)[0, 0].item(),
    'C': model.to_tokens(" C", prepend_bos=False)[0, 0].item(),
    'D': model.to_tokens(" D", prepend_bos=False)[0, 0].item(),
}

print(f"Answer token IDs: {answer_tokens}")

def get_layer_representations_simple(model, tokens):
    """
    Get residual stream representations at each layer using hooks.
    Returns list of representations (each shape [seq_len, d_model]) and cache.
    """
    n_layers = model.cfg.n_layers
    seq_len = tokens.shape[1]
    
    # Store representations
    representations = []
    
    # Define hook function to capture residual stream after each layer
    def cache_hook(resid, hook):
        representations.append(resid.clone())
        return resid
    
    # Create hooks for each layer's residual stream after the layer
    fwd_hooks = []
    for layer in range(n_layers):
        hook_name = f"blocks.{layer}.hook_resid_post"
        fwd_hooks.append((hook_name, cache_hook))
    
    # Run model with hooks
    with t.no_grad():
        logits = model.run_with_hooks(
            tokens,
            return_type="logits",
            fwd_hooks=fwd_hooks,
            prepend_bos=False
        )
    
    # Also get embedding representation (layer -1)
    # We need to get the embedding from a separate forward pass or from cache
    # Let's get it by running without hooks for embeddings
    with t.no_grad():
        embed = model.embed(tokens)
    
    # Insert embedding at beginning
    representations.insert(0, embed)
    
    return representations, logits

def get_layer_unembeddings(representations, model):
    """
    Apply unembedding to each layer's representation.
    """
    layer_logits = []
    for rep in representations:
        # Apply unembedding: [seq_len, d_model] @ [d_model, d_vocab] -> [seq_len, d_vocab]
        logits = rep @ model.W_U
        layer_logits.append(logits)
    return layer_logits

def extract_answer_logits(layer_logits, tokens, answer_tokens):
    """
    Extract logits for answer tokens at the last position.
    """
    last_token_idx = tokens.shape[1] - 1
    n_layers = len(layer_logits)
    n_answers = len(answer_tokens)
    
    answer_logits = np.zeros((n_layers, n_answers))
    answer_names = list(answer_tokens.keys())
    
    for layer_idx, logits in enumerate(layer_logits):
        last_token_logits = logits[last_token_idx]
        for ans_idx, (ans_name, ans_token) in enumerate(answer_tokens.items()):
            answer_logits[layer_idx, ans_idx] = last_token_logits[ans_token].item()
    
    return answer_logits, answer_names

# Tokenize input
tokens = model.to_tokens(wmdp_question, prepend_bos=False)
print(f"Input shape: {tokens.shape}")
print(f"Sequence length: {tokens.shape[1]} tokens")
print(f"Tokens: {model.to_str_tokens(tokens[0])}")

# Get representations and logits
print("\nRunning model and extracting layer representations...")
representations, final_logits = get_layer_representations_simple(model, tokens)
print(f"Number of representations (embed + {len(representations)-1} layers): {len(representations)}")

# Get unembeddings for each layer
layer_logits = get_layer_unembeddings(representations, model)
answer_logits, answer_names = extract_answer_logits(layer_logits, tokens, answer_tokens)

# Get final prediction
final_pred_token = final_logits[0, -1].argmax().item()
predicted_str = model.to_string(final_pred_token)

print(f"\nModel's prediction: '{predicted_str}'")
print(f"Answer logits at final layer:")
for name, logit in zip(answer_names, answer_logits[-1]):
    print(f"  {name}: {logit:.3f}")

# Plot 1: Raw logits through layers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Subplot 1: Raw logits
for i, name in enumerate(answer_names):
    axes[0, 0].plot(range(len(answer_logits)), answer_logits[:, i], marker='o', label=f"Token {name}")
axes[0, 0].set_xlabel("Layer")
axes[0, 0].set_ylabel("Logit Value")
axes[0, 0].set_title("Answer Token Logits Through Layers (Raw)")
axes[0, 0].set_xticks(range(len(answer_logits)))
axes[0, 0].set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(answer_logits)-1)])
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Subplot 2: Softmax probabilities
answer_probs = np.exp(answer_logits) / np.exp(answer_logits).sum(axis=1, keepdims=True)
for i, name in enumerate(answer_names):
    axes[0, 1].plot(range(len(answer_probs)), answer_probs[:, i], marker='o', label=f"Token {name}")
axes[0, 1].set_xlabel("Layer")
axes[0, 1].set_ylabel("Probability")
axes[0, 1].set_title("Answer Token Probabilities Through Layers")
axes[0, 1].set_xticks(range(len(answer_probs)))
axes[0, 1].set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(answer_probs)-1)])
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Subplot 3: Logit difference (predicted vs others)
predicted_idx = np.argmax(answer_logits[-1])
logit_diff_pred_vs_next = answer_logits[:, predicted_idx] - np.delete(answer_logits, predicted_idx, axis=1).max(axis=1)
axes[1, 0].plot(range(len(logit_diff_pred_vs_next)), logit_diff_pred_vs_next, marker='s', color='green', linewidth=2)
axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[1, 0].set_xlabel("Layer")
axes[1, 0].set_ylabel("Logit Difference")
axes[1, 0].set_title(f"Logit({answer_names[predicted_idx]}) - Next Best")
axes[1, 0].set_xticks(range(len(logit_diff_pred_vs_next)))
axes[1, 0].set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(logit_diff_pred_vs_next)-1)])
axes[1, 0].grid(True, alpha=0.3)

# Subplot 4: Logit difference (predicted vs mean)
logit_diff_pred_vs_mean = answer_logits[:, predicted_idx] - np.mean(answer_logits, axis=1)
axes[1, 1].plot(range(len(logit_diff_pred_vs_mean)), logit_diff_pred_vs_mean, marker='s', color='purple', linewidth=2)
axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
axes[1, 1].set_xlabel("Layer")
axes[1, 1].set_ylabel("Logit Difference")
axes[1, 1].set_title(f"Logit({answer_names[predicted_idx]}) - Mean of All")
axes[1, 1].set_xticks(range(len(logit_diff_pred_vs_mean)))
axes[1, 1].set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(logit_diff_pred_vs_mean)-1)])
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot 2: Heatmap
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(answer_logits.T, cmap='RdBu_r', aspect='auto')
ax.set_xticks(range(len(answer_logits)))
ax.set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(answer_logits)-1)])
ax.set_yticks(range(len(answer_names)))
ax.set_yticklabels(answer_names)
ax.set_xlabel("Layer")
ax.set_ylabel("Answer Token")
ax.set_title("Answer Logit Heatmap Through Layers")
plt.colorbar(im, ax=ax, label="Logit Value")
plt.show()

# Plot 3: Layer contributions (how much each layer adds)
layer_contributions = np.diff(answer_logits, axis=0, prepend=answer_logits[:1] * 0)

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(layer_contributions))
width = 0.2

for i, name in enumerate(answer_names):
    ax.bar([xi + i*width for xi in x], layer_contributions[:, i], width, label=f"Token {name}")

ax.set_xlabel("Layer")
ax.set_ylabel("Contribution to Logit")
ax.set_title("Layer Contributions to Answer Token Logits")
ax.set_xticks([xi + width*1.5 for xi in x])
ax.set_xticklabels(["Embed"] + [f"L{i}" for i in range(len(layer_contributions)-1)])
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# Find emergence layer (where probability > threshold)
threshold = 0.5
emergence_layer = None
for layer in range(len(answer_probs)):
    if answer_probs[layer, predicted_idx] > threshold:
        emergence_layer = layer
        break

if emergence_layer is not None:
    print(f"\nAnswer '{answer_names[predicted_idx]}' emerges at layer {emergence_layer} (probability {answer_probs[emergence_layer, predicted_idx]:.3f})")
else:
    print(f"\nAnswer never exceeds {threshold} probability threshold")

# Show token probabilities at key layers
print("\n=== Detailed Layer Analysis ===")
key_layers = [0, len(answer_probs)//3, 2*len(answer_probs)//3, len(answer_probs)-1]
for layer in key_layers:
    print(f"\nLayer {layer} ({'Embed' if layer == 0 else f'L{layer-1}'}):")
    for name, prob in zip(answer_names, answer_probs[layer]):
        print(f"  {name}: {prob:.3f}")

# Optional: Visualize attention at a specific layer
layer_to_viz = 6  # Choose a middle layer
if layer_to_viz < model.cfg.n_layers:
    # Need to run again with cache for attention patterns
    print(f"\nGetting attention patterns for layer {layer_to_viz}...")
    
    attention_patterns = []
    def attn_hook(attn, hook):
        attention_patterns.append(attn.clone())
        return attn
    
    with t.no_grad():
        _ = model.run_with_hooks(
            tokens,
            return_type="logits",
            fwd_hooks=[(f"blocks.{layer_to_viz}.attn.hook_pattern", attn_hook)],
            prepend_bos=False
        )
    
    if attention_patterns:
        str_tokens = model.to_str_tokens(tokens[0])
        attn_to_show = attention_patterns[0][0]  # First batch, all heads
        
        print(f"\nAttention patterns for layer {layer_to_viz}:")
        display(cv.attention.attention_patterns(
            tokens=str_tokens,
            attention=attn_to_show,
            attention_head_names=[f"L{layer_to_viz}H{i}" for i in range(min(12, attn_to_show.shape[0]))]
        ))

print("\n=== Key Findings ===")
print(f"Final answer: {answer_names[predicted_idx]}")
print(f"Final logit: {answer_logits[-1, predicted_idx]:.3f}")
print(f"Final probability: {answer_probs[-1, predicted_idx]:.3f}")
print(f"Margin over next best: {logit_diff_pred_vs_next[-1]:.3f}")
print(f"Answer emerges at layer: {emergence_layer if emergence_layer is not None else 'Never'}")

Answer token IDs: {'A': 362, 'B': 425, 'C': 356, 'D': 422}
Input shape: torch.Size([1, 54])
Sequence length: 54 tokens
Tokens: ['What', ' is', ' the', ' primary', ' mechanism', ' by', ' which', ' transformer', ' models', ' perform', ' in', '-context', ' learning', '?\n\n', 'A', ')', ' Ind', 'uction', ' heads', ' that', ' attend', ' to', ' repeated', ' patterns', '\n', 'B', ')', ' Position', 'al', ' enc', 'odings', ' that', ' preserve', ' sequence', ' order', '\n', 'C', ')', ' Res', 'idual', ' connections', ' that', ' preserve', ' gradient', ' flow', '\n', 'D', ')', ' Layer', ' normalization', ' that', ' stabil', 'izes', ' training']

Running model and extracting layer representations...
Number of representations (embed + 36 layers): 37


IndexError: index 53 is out of bounds for dimension 0 with size 1